# Pitcher workload entering each appearance

Inspect and validate calendar-day workload derived from the reusable pitcher-appearance view. Every workload field describes dates before the current appearance; the current game's pitches and ups are excluded.

## 1. Setup and parameters

In [1]:
%load_ext autoreload
%autoreload 2

from crooked_numbers_analytics.appearances import (
    create_pitcher_appearances_view,
)
from crooked_numbers_analytics.db import open_database
from crooked_numbers_analytics.workload import create_pitcher_workload_view

con = open_database()

In [2]:
season = 2024
team = "MIL"
pitcher_id = 642207  # Devin Williams; change to another MLBAM pitcher ID

## 2. Build the reusable views

The appearance helper reads only the selected season's local Hive partitions. Workload is then calculated from the much smaller appearance-level relation.

In [3]:
create_pitcher_appearances_view(con, seasons=[season])
create_pitcher_workload_view(con)

## 3. Inspect one pitcher's workload

The windows are prior calendar days: for June 10, previous day is June 9, last two days are June 8-9, and last seven days are June 3-9. `days_rest` counts full off-days between the prior appearance date and the current date.

In [4]:
con.execute(
    """
    SELECT
        game_date, game_pk, pitching_team, pitcher_id, pitcher_name,
        is_start, pitch_count, ups, previous_appearance_date, days_rest,
        pitches_previous_day, pitches_last_2_days, pitches_last_3_days,
        pitches_last_7_days, ups_previous_day, ups_last_3_days,
        appearances_previous_day, appearances_last_3_days,
        pitched_previous_day, pitched_two_consecutive_days
    FROM pitcher_workload
    WHERE season = ? AND pitcher_id = ?
    ORDER BY game_date, game_pk
    LIMIT 100
    """,
    [season, pitcher_id],
).df()

,game_date,game_pk,pitching_team,pitcher_id,pitcher_name,is_start,pitch_count,ups,previous_appearance_date,days_rest,pitches_previous_day,pitches_last_2_days,pitches_last_3_days,pitches_last_7_days,ups_previous_day,ups_last_3_days,appearances_previous_day,appearances_last_3_days,pitched_previous_day,pitched_two_consecutive_days
0,2024-07-28,745961,MIL,642207,"Williams, Devin",False,24,1,NaT,<NA>,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,False,False
1,2024-08-04,744826,MIL,642207,"Williams, Devin",False,17,1,2024-07-28,6,0.0,0.0,0.0,24.0,0.0,0.0,0.0,0.0,False,False
2,2024-08-07,747085,MIL,642207,"Williams, Devin",False,13,1,2024-08-04,2,0.0,0.0,17.0,17.0,0.0,1.0,0.0,1.0,False,False
3,2024-08-10,745955,MIL,642207,"Williams, Devin",False,18,1,2024-08-07,2,0.0,0.0,13.0,30.0,0.0,1.0,0.0,1.0,False,False
4,2024-08-14,745951,MIL,642207,"Williams, Devin",False,7,1,2024-08-10,3,0.0,0.0,0.0,31.0,0.0,0.0,0.0,0.0,False,False
5,2024-08-15,745950,MIL,642207,"Williams, Devin",False,16,1,2024-08-14,0,7.0,7.0,7.0,25.0,1.0,1.0,1.0,1.0,True,False
6,2024-08-17,745952,MIL,642207,"Williams, Devin",False,10,1,2024-08-15,1,0.0,16.0,23.0,41.0,0.0,2.0,0.0,2.0,False,False
7,2024-08-20,745136,MIL,642207,"Williams, Devin",False,21,1,2024-08-17,2,0.0,0.0,10.0,33.0,0.0,1.0,0.0,1.0,False,False
8,2024-08-21,745138,MIL,642207,"Williams, Devin",False,31,1,2024-08-20,0,21.0,21.0,21.0,54.0,1.0,1.0,1.0,1.0,True,False
9,2024-08-28,745943,MIL,642207,"Williams, Devin",False,11,1,2024-08-21,6,0.0,0.0,0.0,31.0,0.0,0.0,0.0,0.0,False,False


## 4. Validate rolling windows

For a small pitcher sample, independently sum prior appearance rows and compare them with the reusable three-day fields. Empty results are expected.

In [5]:
con.execute(
    """
    SELECT
        workload.game_date,
        workload.game_pk,
        workload.pitches_last_3_days,
        workload.ups_last_3_days,
        workload.appearances_last_3_days,
        coalesce(sum(history.pitch_count), 0) AS checked_pitches,
        coalesce(sum(history.ups), 0) AS checked_ups,
        count(history.game_pk) AS checked_appearances
    FROM pitcher_workload AS workload
    LEFT JOIN pitcher_appearances AS history
        ON history.pitcher_id = workload.pitcher_id
        AND history.game_date BETWEEN workload.game_date - INTERVAL 3 DAYS
                                  AND workload.game_date - INTERVAL 1 DAY
    WHERE workload.season = ? AND workload.pitcher_id = ?
    GROUP BY ALL
    HAVING workload.pitches_last_3_days <> checked_pitches
        OR workload.ups_last_3_days <> checked_ups
        OR workload.appearances_last_3_days <> checked_appearances
    ORDER BY workload.game_date, workload.game_pk
    """,
    [season, pitcher_id],
).df()

,game_date,game_pk,pitches_last_3_days,ups_last_3_days,appearances_last_3_days,checked_pitches,checked_ups,checked_appearances


## 5. Inspect multi-inning appearances and ups

An up is one distinct inning with a recorded pitch. It is not inferred from the span between first and last inning, and it does not estimate warmups that occurred without entering a game.

In [6]:
con.execute(
    """
    SELECT
        game_date, game_pk, pitcher_id, pitcher_name, is_start,
        pitch_count, first_inning, last_inning, innings_appeared, ups
    FROM pitcher_workload
    WHERE season = ? AND pitching_team = ? AND ups > 1
    ORDER BY ups DESC, pitch_count DESC, game_date
    LIMIT 50
    """,
    [season, team],
).df()

,game_date,game_pk,pitcher_id,pitcher_name,is_start,pitch_count,first_inning,last_inning,innings_appeared,ups
0,2024-07-10,745969,668964,"Myers, Tobias",True,103,1,8,8,8
1,2024-06-07,746466,668964,"Myers, Tobias",True,100,1,8,8,8
2,2024-08-10,745955,668964,"Myers, Tobias",True,99,1,8,8,8
3,2024-08-18,745948,607067,"Rea, Colin",True,88,1,8,8,8
4,2024-06-30,745971,642547,"Peralta, Freddy",True,103,1,7,7,7
5,2024-08-24,745617,607067,"Rea, Colin",True,102,1,7,7,7
6,2024-07-11,745967,650644,"Civale, Aaron",True,101,1,7,7,7
7,2024-05-22,746063,642547,"Peralta, Freddy",True,98,1,7,7,7
8,2024-07-03,746534,607067,"Rea, Colin",True,97,1,7,7,7
9,2024-09-14,747148,668964,"Myers, Tobias",True,97,1,7,7,7


In [7]:
con.execute(
    """
    SELECT
        is_start, ups, COUNT(*) AS appearances,
        round(avg(pitch_count), 1) AS average_pitches
    FROM pitcher_workload
    WHERE season = ? AND pitching_team = ?
    GROUP BY is_start, ups
    ORDER BY is_start DESC, ups
    """,
    [season, team],
).df()

,is_start,ups,appearances,average_pitches
0,True,1,6,11.8
1,True,2,5,20.8
2,True,3,4,69.3
3,True,4,24,74.6
4,True,5,46,86.5
5,True,6,58,90.8
6,True,7,15,92.7
7,True,8,4,97.5
8,False,1,343,14.7
9,False,2,137,25.5


## 6. Basic sanity checks

The rolling windows should be nondecreasing as they widen, dates should move forward, and workload counts should never be negative. Empty results are expected.

In [8]:
con.sql(
    """
    SELECT *
    FROM pitcher_workload
    WHERE previous_appearance_date >= game_date
       OR days_rest < 0
       OR pitches_previous_day > pitches_last_2_days
       OR pitches_last_2_days > pitches_last_3_days
       OR pitches_last_3_days > pitches_last_7_days
       OR ups_previous_day > ups_last_2_days
       OR ups_last_2_days > ups_last_3_days
       OR ups_last_3_days > ups_last_7_days
       OR appearances_previous_day > appearances_last_2_days
       OR appearances_last_2_days > appearances_last_3_days
       OR appearances_last_3_days > appearances_last_7_days
    ORDER BY game_date, game_pk
    LIMIT 100
    """
).df()

,game_pk,game_date,season,pitching_team,pitcher_id,pitcher_name,is_start,pitch_count,first_inning,last_inning,...,ups_previous_day,ups_last_2_days,ups_last_3_days,ups_last_7_days,appearances_previous_day,appearances_last_2_days,appearances_last_3_days,appearances_last_7_days,pitched_previous_day,pitched_two_consecutive_days


In [9]:
con.sql(
    """
    SELECT
        COUNT(*) AS appearances,
        COUNT(DISTINCT (game_pk, pitching_team, pitcher_id)) AS unique_keys,
        COUNT(*) FILTER (WHERE pitcher_id IS NULL) AS null_pitcher_ids,
        COUNT(*) FILTER (WHERE pitch_count <= 0) AS nonpositive_pitch_counts,
        COUNT(*) FILTER (WHERE ups <= 0) AS nonpositive_ups
    FROM pitcher_workload
    """
).df()

,appearances,unique_keys,null_pitcher_ids,nonpositive_pitch_counts,nonpositive_ups
0,20693,20693,0,0,0


## Same-day appearance behavior

Calendar-day workload deliberately excludes the entire current date. If a pitcher appears twice in a doubleheader, both rows have identical workload from earlier dates; the two appearances are aggregated together for the following day's windows. Statcast does not provide a reliable cross-game start time here, so same-day game ordering remains future work.